[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 05](README.md)

# OpenMP target: datos, reducción y fallback

**Tema:** 05 · **Sesiones:** 23, 24 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo minimizar mapeos sin perder coherencia entre host y dispositivo?


## Resultados de aprendizaje

- Explicar `target`, `teams` y `distribute parallel for`.
- Elegir map/to/from/alloc según el flujo de datos.
- Validar reducción en dispositivo y fallback.


## Modelo conceptual

Las regiones de datos persistentes evitan transferencias repetidas cuando varias operaciones reutilizan arreglos.

`map(to:)` inicializa en dispositivo, `from:` recupera y `tofrom:` realiza ambas direcciones.

Una reducción requiere soporte del compilador/runtime y se valida con una referencia numérica.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "05"
NOTEBOOK = "05_openmp_target/02_openmp_target.ipynb"
assert (ROOT / "curso" / "notebooks" / "05_openmp_target" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Contrato de mapeo

Se deriva la dirección mínima para entradas y salidas de vector add.


In [ ]:
arrays = {"a": {"read": True, "write": False}, "b": {"read": True, "write": False}, "c": {"read": False, "write": True}}
def clause(access):
    if access["read"] and access["write"]: return "tofrom"
    if access["read"]: return "to"
    if access["write"]: return "from"
    return "alloc"
mapping = {name: clause(access) for name, access in arrays.items()}
assert mapping == {"a": "to", "b": "to", "c": "from"}
print(mapping)


**Interpretación.** El contrato se revisa cuando un arreglo persiste entre kernels o se inicializa en el dispositivo.


## Fallback correcto

Se ejecuta una referencia portable de vector add y se valida elemento a elemento.


In [ ]:
n = 1000
a = [i * 0.5 for i in range(n)]
b = [1.0 - i * 0.25 for i in range(n)]
c = [x + y for x, y in zip(a, b)]
expected = [1.0 + i * 0.25 for i in range(n)]
error = max(abs(x-y) for x, y in zip(c, expected))
assert error < 1e-12
print({"n": n, "max_error": error, "path": "modelo CPU para validar el kernel target"})


**Interpretación.** La misma entrada y tolerancia se reutilizan cuando el kernel OpenMP target se ejecuta en hardware real.


## Práctica reproducible

1. Implementar vector add con región de datos explícita.
2. Reutilizar datos durante varias operaciones y medir ambas variantes.
3. Registrar compilador, plugin de offload y dispositivo.


## Errores frecuentes

- Mapear `tofrom` todo por comodidad.
- Acceder en host antes de sincronizar.
- Declarar éxito GPU sin comprobar `omp_is_initial_device`.

## Criterios de aceptación

- Mapeo mínimo justificado.
- Fallback y dispositivo producen resultados equivalentes.
- Informe identifica inequívocamente dónde se ejecutó.


## Referencias y material relacionado

- [Planeación target](../../../docs/PLANEACION_CURSO.md)
- [Protocolo](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 05](README.md)
